# Setup and checks

**What it does**
Everything needed to get KCT running on a machine, one cell per step, no terminal: checks Python and the packages, installs what is missing, shows the active settings, asks the AI "What is 5+2?", checks that SQLite works, runs the tests, and starts and stops the API.

**Why**
So you can set up and check the project in JupyterLab (in ING: on the Vertex AI Workbench) by pressing ▶ on each cell. It does the same as `kct.sh` in a terminal.

**How to use**
1. Open this notebook from the project's `notebooks/` folder.
2. Fill in **Step 1** (settings). For the first try leave it as it is: the provider is then `fake`, which needs no cloud access. Run **Step 1b** to save the settings to `.env` for the other notebooks and the API.
3. Run the cells one by one with ▶ (or Shift+Enter), top to bottom. Each cell says what it checked; ✅ is good, ❌ tells you what to do.
4. After **Step 3** installs anything, restart the kernel (Kernel > Restart Kernel) and run again from Step 1.
5. Every step can be run again at any time. Skip the ones you do not need.

## Step 1: Settings

Finds the project folder and sets the configuration. Settings come from the project's `.env` file, if there is one, and from `SETTINGS` below, which wins over `.env`. No `.env` is needed: the cell is enough. With neither, the provider is `fake`.

- `KCT_AI_PROVIDER`: `fake` (no cloud, fixed answers), `gemini` or `anthropic`.
- `KCT_PROJECT_ID`: your Google Cloud project, needed for `gemini` and `anthropic`.
- `PIP_INDEX`: the internal pip mirror URL, if ING has one; leave empty otherwise.

Settings are read once per kernel: after changing them here, restart the kernel and run from this cell again.

In [ ]:
import os, sys
from pathlib import Path

# The project folder: the parent of notebooks/. Change it if you moved the notebook.
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Uncomment and fill in what you need. These win over .env.
SETTINGS = {
    # "KCT_AI_PROVIDER": "gemini",  # fake | gemini | anthropic
    # "KCT_PROJECT_ID": "<your-project>",
    # "KCT_LOCATION": "europe-west4",
    # "KCT_DB_DRIVER": "memory",  # memory | sqlite | bigquery
}
PIP_INDEX = ""  # internal pip mirror URL, if any

from services.config import load_env_file

load_env_file()                                  # the project's .env, if there is one
os.environ.update(SETTINGS)                      # this cell wins over .env
os.environ.setdefault("KCT_AI_PROVIDER", "fake")  # nothing chosen anywhere: no cloud
print(".env:", "loaded" if (PROJECT / ".env").is_file() else "none, using the settings in this cell")
print("AI provider:", os.environ["KCT_AI_PROVIDER"])
print("Project folder:", PROJECT)

## Step 1b: Save the settings to `.env` (optional)

Writes the values from `SETTINGS` in Step 1 into the project's `.env` file, so every notebook, `kct.sh` and the API use them, and you do not have to edit `.env` in vim. If there is no `.env` yet it starts from `.env.example`. Lines you did not put in `SETTINGS` stay as they are.

Run Step 1 first. Restart the kernel afterwards so everything reads the new file.

To look at or edit `.env` by hand without vim: in JupyterLab use **File > Open from Path...**, type `.env`, and it opens in JupyterLab's own text editor (save with Ctrl+S). The file browser hides files starting with a dot unless you turn on **View > Show Hidden Files**.

In [ ]:
import re, shutil

env_path = PROJECT / ".env"
if not env_path.is_file():
    shutil.copy(PROJECT / ".env.example", env_path)
    print("Created .env from .env.example")

lines = env_path.read_text(encoding="utf-8").splitlines()
# The provider in use is always saved, so .env.example's default cannot switch it.
to_save = {"KCT_AI_PROVIDER": os.environ["KCT_AI_PROVIDER"], **SETTINGS}
for key, value in to_save.items():
    pattern = re.compile(rf"^\s*#?\s*{re.escape(key)}=")
    for i, line in enumerate(lines):
        if pattern.match(line):
            lines[i] = f"{key}={value}"
            break
    else:
        lines.append(f"{key}={value}")
env_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("Saved to", env_path)
for line in lines:
    if line.strip() and not line.lstrip().startswith("#"):
        print("  ", line)
print("\nRestart the kernel (Kernel > Restart Kernel) so everything reads the new .env.")

## Step 2: Check Python and packages

Shows the Python version and, for each package, whether it is installed. Only the packages in the "notebooks" group are needed to use the notebooks with Gemini; the others are for the API or for Claude.

In [ ]:
import importlib.util, importlib.metadata, sqlite3

PACKAGES = {
    # group: [(import name, pip name)]
    "notebooks": [("pydantic", "pydantic>=2"), ("google.genai", "google-genai"), ("pandas", "pandas"),
                  ("docx", "python-docx"), ("reportlab", "reportlab"), ("openpyxl", "openpyxl"),
                  ("extract_msg", "extract-msg"), ("pypdf", "pypdf")],
    "api": [("fastapi", "fastapi"), ("uvicorn", "uvicorn"), ("multipart", "python-multipart"),
            ("httpx", "httpx")],
    "claude": [("anthropic", "anthropic[vertex]")],
}

def installed(module, requirement):
    if importlib.util.find_spec(module.split(".")[0]) is None or importlib.util.find_spec(module) is None:
        return False
    if module == "pydantic":
        return importlib.metadata.version("pydantic").startswith("2")
    return True

ok_python = sys.version_info >= (3, 9)
print(("✅" if ok_python else "❌ needs Python 3.9+:"), "Python", sys.version.split()[0], "-", sys.executable)
print("✅ sqlite3", sqlite3.sqlite_version, "(built into Python, nothing to install)")
for group, packages in PACKAGES.items():
    print(f"\n{group}:")
    for module, requirement in packages:
        print("  ✅" if installed(module, requirement) else "  ❌ missing", requirement)

## Step 3: Install what is missing

Installs only the missing packages, into the Python this notebook runs on. Choose the groups below. Nothing is installed when everything is already there.

**After it installs anything: restart the kernel (Kernel > Restart Kernel) and run again from Step 1.**

In [ ]:
import subprocess

INSTALL = ["notebooks", "api"]   # "api": the API and the tests; add "claude" for Claude
missing = [req for group in INSTALL for module, req in PACKAGES[group] if not installed(module, req)]

if not missing:
    print("✅ Nothing to install")
else:
    command = [sys.executable, "-m", "pip", "install", *missing]
    if PIP_INDEX:
        command += ["--index-url", PIP_INDEX]
    print("Installing:", " ".join(missing))
    result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout[-2000:], result.stderr[-2000:])
    if result.returncode == 0:
        print("✅ Installed. Now restart the kernel (Kernel > Restart Kernel) and run from Step 1.")
    else:
        print("❌ pip failed (see above). No internet? Ask for the internal pip mirror and set PIP_INDEX in Step 1.")

## Step 4: Show the active settings

What the app will use: AI provider and model, Google Cloud project, and where files and data go. `memory` as database means nothing is stored in a database.

In [ ]:
from services.config import settings

config = settings()
for name in ["ai_provider", "ai_model", "gcp_project", "gemini_location", "files_driver", "db_driver", "storage_root"]:
    print(f"{name:<16} {getattr(config, name) or '<not set>'}")

if config.ai_provider != "fake" and not config.gcp_project and config.anthropic_platform == "vertex":
    print("\n❌ KCT_PROJECT_ID is not set; Gemini and Claude on Vertex AI need it (Step 1).")

## Step 5: Ask the AI "What is 5+2?"

The smoke test: one question through the app's `AskService`, the same code the API's `POST /api/v1/ask` runs. With `fake` the answer is always `5 + 2 = 7.`; with a real provider you also see which model answered, the tokens used and the time.

In [ ]:
from services.app import boot

kct = boot()
try:
    result = kct.ask.ask("What is 5+2?")
except Exception as exc:
    print("❌", type(exc).__name__, exc)
else:
    print("✅", result.answer)
    print(f"   {result.provider}/{result.model}, {result.input_tokens} in / {result.output_tokens} out tokens, {result.duration_ms} ms")

## Step 6: Check SQLite works

Writes one record into a separate test file, `storage/sqlite_test.sqlite`, and reads the last five back. It never touches the app's data. For a longer walk-through see `03_sqlite_test.ipynb`.

In [ ]:
db = PROJECT / "storage" / "sqlite_test.sqlite"
db.parent.mkdir(parents=True, exist_ok=True)
with sqlite3.connect(db) as con:
    con.execute("CREATE TABLE IF NOT EXISTS setup_check (id INTEGER PRIMARY KEY AUTOINCREMENT, "
                "name TEXT NOT NULL, created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP)")
    con.execute("INSERT INTO setup_check (name) VALUES (?)", ("written by 01_setup.ipynb",))
    rows = con.execute("SELECT id, name, created_at FROM setup_check ORDER BY id DESC LIMIT 5").fetchall()

print("✅ SQLite", sqlite3.sqlite_version, "works:", db)
for row in rows:
    print("  ", *row)

## Step 7: Run the tests (optional)

Runs the project's test suite with the fake AI in a temporary folder, so it costs nothing and leaves no files. Takes a few seconds. Needs the "api" packages (Step 3); tests for packages that are not installed (Claude, Google Cloud Storage, BigQuery) are skipped, not failed.

In [ ]:
# PYTHONUTF8 and encoding="utf-8": on Windows, output is otherwise read as cp1252 and ✅ breaks it.
# The tests run without this notebook's KCT_* settings, so Step 1 cannot change them.
test_env = {k: v for k, v in os.environ.items() if not k.startswith("KCT_")} | {"PYTHONUTF8": "1", "KCT_ENV_FILE": ""}
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "tests"],
                        capture_output=True, text=True, encoding="utf-8", errors="replace",
                        cwd=PROJECT, env=test_env)
print(result.stderr[-3000:])
print("✅ All tests pass" if result.returncode == 0 else "❌ Some tests failed or could not start (see above)")

## Step 8: Start the API (optional)

Starts the API in the background on `API_PORT` and checks it answers `/health` and `/ask`. It keeps running until you run Step 9 or restart the kernel. Needs the "api" packages (Step 3 with `"api"`).

On a Vertex AI Workbench the docs page opens at `https://<workbench-url>/proxy/<API_PORT>/docs`, if your browser may open it.

In [ ]:
import json, time, urllib.request

API_PORT = 8000

def call(path, body=None):
    request = urllib.request.Request(
        f"http://localhost:{API_PORT}/api/v1{path}",
        data=None if body is None else json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    return json.loads(urllib.request.urlopen(request, timeout=300).read())

api_log = open(PROJECT / "storage" / "api_from_notebook.log", "w", encoding="utf-8")
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", str(API_PORT)],
                       cwd=PROJECT, env={**os.environ, "PYTHONUTF8": "1"}, stdout=api_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("✅ API running:", call("/health"))
        print("✅ /ask:", call("/ask", {"question": "What is 5+2?"})["answer"])
        break
    except Exception:
        if api.poll() is not None:
            print("❌ API stopped; see storage/api_from_notebook.log. Port taken? Change API_PORT.")
            break
        time.sleep(0.5)
else:
    print("❌ API did not answer within 30 s; see storage/api_from_notebook.log")

## Step 9: Stop the API

In [ ]:
try:
    api.terminate()
    api.wait(timeout=10)
    print("✅ API stopped")
except NameError:
    print("The API was not started from this notebook")